# 微調預訓練 BERT：影評情緒分類（Rotten Tomatoes）

---

## 這份程式要做什麼

任務：給一句影評，判斷它是**正面**還是**負面**（二元文字分類）。

方法：**微調（fine-tuning）** 一個已經預訓練好的 BERT。

```
          Google 預訓練好的 bert-base-cased
          （已經懂英文語法與詞義）
                    │
        ① 載入權重，並在最上面接一個新的分類層（2 類）
                    │
        ② 把影評文字轉成數字（tokenize）
                    │
        ③ 用 8500 多筆標註好的影評，以小學習率繼續訓練
                    │
        ④ 用 F1 分數評估，訓練前後對照
                    ▼
           一個專門判斷影評情緒的模型
```

## 什麼是「微調」

| 做法 | 起點 | 需要的資料量 |
|---|---|---|
| 從頭訓練 | 隨機權重 | 非常大 |
| **微調（這份）** | 預訓練權重 | 小（幾千筆就有效果） |

微調 = 站在巨人的肩膀上：BERT 已經讀過海量文字，我們只需要用少量資料，把它「往我們的任務微微調整」。

## 關於「微調到底發生在哪裡」

很多初學者看完程式會疑惑：「我沒看到哪一行寫著微調。」答案是它由**三件事的組合**構成：

1. `from_pretrained`：起點是預訓練好的權重
2. `num_labels = 2`：在 BERT 上接一個新的分類頭
3. `trainer.train()`：用任務資料更新權重

下面逐步看。

## 步驟 0：載入套件

| 套件 / 類別 | 用途 |
|---|---|
| `datasets.load_dataset` | 下載並載入 Hugging Face 上的資料集 |
| `AutoTokenizer` | 自動載入對應模型的分詞器（把文字切成 token 並轉成數字） |
| `AutoModelForSequenceClassification` | 自動載入「BERT + 分類頭」的模型 |
| `DataCollatorWithPadding` | 把一個 batch 的句子補齊成相同長度 |
| `evaluate` | Hugging Face 的評估指標套件（這裡用 F1） |
| `TrainingArguments`、`Trainer` | 設定訓練參數、執行訓練與評估 |
| `numpy` | 數值運算（這裡用 `argmax`） |

名稱裡的 **Auto** 的意思是：你只要給模型名稱，它會自動判斷該用哪一種模型架構，不用自己指定是 BERT 還是別的。

In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import DataCollatorWithPadding
import numpy as np
import evaluate
from transformers import TrainingArguments, Trainer

## 步驟 1：載入資料集

Rotten Tomatoes 是影評資料集，每筆資料是一句影評加上一個標籤：

| 欄位 | 說明 |
|---|---|
| `text` | 一句影評 |
| `label` | `0` = 負面，`1` = 正面 |

資料集已經切好三份：`train`、`validation`、`test`。

```python
train_data, test_data = tomatoes["train"], tomatoes["test"]
```

用 `[ ]` 取出 train 和 test 兩份。這裡**沒有用到 validation**。

In [2]:
tomatoes = load_dataset("cornell-movie-review-data/rotten_tomatoes")
train_data, test_data = tomatoes["train"], tomatoes["test"]

## 步驟 2：載入預訓練模型與分詞器（微調的起點）

```python
model_id = "google-bert/bert-base-cased"
```

`bert-base-cased` 是 Google 預訓練的 BERT，**區分大小寫**（`Apple` 和 `apple` 是不同的詞），12 層、約 1.08 億個參數。

### 模型

```python
AutoModelForSequenceClassification.from_pretrained(model_id, num_labels = 2)
```

這一行同時做了兩件事：

| 部分 | 來源 | 狀態 |
|---|---|---|
| BERT 本體（12 層） | 從 `model_id` 載入 | **已訓練好**的權重 |
| 最上面的分類層（輸出 2 個分數） | `num_labels = 2` 新建 | **隨機初始化**，還沒學過 |

載入時你會看到類似這樣的警告，這是**正常的**，不是錯誤：

```
Some weights of BertForSequenceClassification were not initialized from the
model checkpoint ... and are newly initialized: ['classifier.weight', 'classifier.bias']
```

意思是：分類層是新加的，等一下要靠訓練來學。

### 分詞器

`AutoTokenizer.from_pretrained(model_id)` 載入和 BERT 配套的分詞器。**模型和分詞器一定要配對**，因為每個模型有自己的詞彙表，用錯會導致數字對不上。

In [3]:
# First Step : 載入基礎預訓練模型
model_id = "google-bert/bert-base-cased"
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels = 2,
)
tokenizer = AutoTokenizer.from_pretrained(
    model_id
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 步驟 3：分詞（tokenize）與批次補齊（padding）

模型只認得數字，所以要把文字轉成數字。

### 分詞做了什麼

```
"a gorgeous film"
      ↓ 分詞器
[101, 170, 12474, 1273, 102]      ← input_ids（每個 token 對應一個編號）
```

分詞器的輸出還包含 `token_type_ids` 和 `attention_mask`，由 BERT 的流程使用。

### `lambda x : tokenizer(x["text"])` 怎麼讀

這是一個沒有名字的小函數：**丟一批資料 x 進去，吐出分詞後的結果**。

```python
def f(x):
    return tokenizer(x["text"])
```

兩者等價。`batched = True` 讓 `.map()` 一次傳進**一批**資料（不是一筆），此時 `x["text"]` 是一個「句子清單」，分詞器可以一次處理整批，速度快很多。

### 為什麼分詞時不補齊（padding）

句子長短不一，但一個 batch 裡的句子必須等長才能組成矩陣。這份程式把補齊的工作留給 `DataCollatorWithPadding`，它在**組成每個 batch 的時候**才補：

| 做法 | 補到多長 |
|---|---|
| 分詞時全部補到固定最大長度 | 每筆都很長，浪費運算 |
| **`DataCollatorWithPadding`（動態補齊）** | 只補到「這個 batch 裡最長那句」的長度，省時間 |

補上去的位置，`attention_mask` 會標記為 0，模型就知道「這裡是填充，不要看」。

> 這份程式的分詞沒有設 `truncation=True`。影評都是短句，一般不會超過 BERT 的 512 token 上限，所以沒問題；處理長文章時就要加上截斷。

In [4]:
#對資料進行分詞處理
#將多個獨立的資料樣本 組裝成一個 Batch（批次），以供機器學習模型
data_collator = DataCollatorWithPadding(tokenizer= tokenizer)

tokenized_train = train_data.map(lambda x : tokenizer(x["text"]), batched = True)
tokenized_test  =test_data.map(lambda x : tokenizer(x["text"]), batched = True)

## 步驟 4：定義評估指標（compute_metrics）

Trainer 評估時，會呼叫你提供的 `compute_metrics`，把模型的預測交給它，由它算出分數。

### 輸入是什麼

```python
logits, labels = eval_pred
```

| 變數 | 形狀 | 內容 |
|---|---|---|
| `logits` | `(資料筆數, 2)` | 每筆資料兩個**原始分數**（還沒轉成機率）：負面分數、正面分數 |
| `labels` | `(資料筆數,)` | 每筆的正確答案（0 或 1） |

### 逐行說明

| 程式 | 作用 |
|---|---|
| `np.argmax(logits, axis = -1)` | 每筆資料取分數較大的那一個的**索引**，也就是預測類別。例如 `[0.3, 1.2]` → `1`（預測為正面） |
| `evaluate.load("f1")` | 載入 F1 指標 |
| `.compute(predictions=..., references=...)` | 比對預測和正確答案，算出 F1 |
| `return {"f1" : ...}` | **必須回傳 dict**。Trainer 會自動加上前綴，所以輸出裡看到的是 `eval_f1` |

### 什麼是 F1

F1 同時考慮兩件事：

- **Precision（精確率）**：預測為正面的，有多少真的是正面
- **Recall（召回率）**：真正的正面，被抓出來多少

F1 是兩者的調和平均，越高越好。

### 一個重要的基準

測試集正負各半，如果模型**不管看到什麼都猜「正面」**：precision = 0.5、recall = 1，F1 ≈ **0.667**。所以 F1 接近 0.667，代表模型根本沒學到東西，只是在亂猜（或固定猜一邊）。

> 想知道 `compute_metrics` 會收到什麼？先寫一個只會 `print(type(eval_pred))`、印出形狀和前幾筆內容的版本，看過再寫邏輯。這比死背文件可靠得多。

In [5]:
#定義度量指標
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis = -1)

    load_f1 = evaluate.load("f1")
    # print(load_f1)
    f1_scores = load_f1.compute(predictions=predictions, references=labels)["f1"]
    return {"f1" : f1_scores}

## 步驟 5：設定訓練參數（TrainingArguments）

| 參數 | 值 | 說明 |
|---|---|---|
| `output_dir` | `"final_model"` | 訓練輸出資料夾，存放 checkpoint |
| `learning_rate` | `2e-5` | 學習率，**很小**：微調只做小幅調整，避免破壞預訓練學到的知識 |
| `per_device_train_batch_size` | 16 | 訓練時每步用 16 筆 |
| `per_device_eval_batch_size` | 16 | 評估時每次處理 16 筆（只影響速度和記憶體） |
| `num_train_epochs` | 1 | 整份訓練資料看 1 遍 |
| `weight_decay` | 0.01 | 輕微的權重衰減（正則化），防止過擬合 |
| `save_strategy` | `"epoch"` | 每個 epoch 結束存一次 checkpoint |
| `report_to` | `"none"` | 不把記錄傳到 wandb、TensorBoard 等外部工具 |

### 步數速算

訓練資料約 8530 筆 ÷ batch 16 ≈ **534 步**（1 個 epoch）。

### `output_dir` 提醒

這個資料夾放的是**訓練過程的輸出**，不保證根目錄有可直接載入的最終模型。想保存最終模型，訓練完要另外呼叫 `trainer.save_model(...)`。

In [6]:
args = TrainingArguments(
    output_dir = "Finetune_BERT_model",
    learning_rate = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    num_train_epochs = 1,
    weight_decay = 0.01,
    save_strategy = "epoch",
    report_to = "none",
)

## 步驟 6：建立 Trainer

`Trainer` 把所有零件組裝在一起，之後只要呼叫 `train()` 或 `evaluate()`：

| 參數 | 說明 |
|---|---|
| `model` | 要訓練的模型（BERT + 分類頭） |
| `args` | 剛剛設定的訓練參數 |
| `data_collator` | 負責把每個 batch 補齊 |
| `train_dataset` | 訓練資料（已分詞） |
| `eval_dataset` | 評估資料（這裡用 test 集） |
| `compute_metrics` | 評估指標函式 |

> **建立 Trainer 不會自動開始訓練。** 必須明確呼叫 `trainer.train()`，這是初學者最常漏掉的一步。

In [7]:
trainer = Trainer(
    model = model,
    args = args,
    data_collator = data_collator,
    train_dataset = tokenized_train,
    eval_dataset = tokenized_test,
    compute_metrics = compute_metrics,
)

## 步驟 7：訓練前後對照（真正的微調在這裡）

```python
print("Before training:", trainer.evaluate())
trainer.train()
result = trainer.evaluate()
print(f"After training : {result} ")
```

| 呼叫 | 作用 |
|---|---|
| `trainer.evaluate()` | **只打分數**，不會更新任何權重 |
| `trainer.train()` | **真正的微調**，更新模型權重 |

### `trainer.train()` 內部每個 batch 做的事

```
1. 把 16 筆影評丟進 BERT → 分類頭 → 得到預測
2. 和正確標籤計算 loss（交叉熵）
3. 反向傳播，算出每個權重該怎麼調整
4. 優化器（AdamW）以 learning_rate = 2e-5 更新權重
```

**更新的是整個模型的權重**，包含 BERT 本體 12 層和新的分類頭，這稱為「全參數微調」。

### 預期看到什麼

| 時間點 | 預期現象 |
|---|---|
| Before training | 分類頭是隨機的，F1 約 0.667（全猜同一邊），`epoch: 0` |
| After training | F1 明顯提高，`epoch: 1.0` |

訓練後的具體數字我沒有實際執行過，以你跑出來的為準。

### 之前常見的錯誤

如果只有 `trainer.evaluate()` 而沒有 `trainer.train()`，印出來的 "After training" 其實是**還沒訓練的模型**，分數會和 Before 幾乎一樣，輸出裡的 `epoch` 也會是 0。

In [8]:
print("Before training:", trainer.evaluate())
trainer.train()   
trainer.save_model()     
result = trainer.evaluate()
print(f"After training : {result} ")

Training Loss,Validation Loss,Step,F1
No log,0.756373,0,0.666667


Before training: {'eval_loss': 0.7563725113868713, 'eval_f1': 0.6666666666666666}


Step,Training Loss
500,0.411643


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Step,F1
0.411643,0.369371,534,0.855796


After training : {'eval_loss': 0.3693709373474121, 'eval_f1': 0.8557964184731386} 


---

# 重點整理

## 一、整份程式一句話

> 載入預訓練好的 BERT，接上新的二分類頭，用 8500 多筆影評以小學習率訓練一個 epoch，並用 F1 分數對照訓練前後。

## 二、微調 = 三件事的組合

| # | 程式 | 做了什麼 |
|---|---|---|
| ① | `from_pretrained(model_id, ...)` | 起點是**預訓練權重**，不是隨機的 |
| ② | `num_labels = 2` | 在 BERT 上接一個**新的、隨機初始化**的分類頭 |
| ③ | `trainer.train()` | 用任務資料、小學習率，**更新全部權重** |

## 三、必須搞懂的 8 個概念

| # | 概念 | 一句話解釋 |
|---|---|---|
| 1 | 預訓練 | 在海量文字上先學好語言知識 |
| 2 | 微調 | 用少量任務資料，把預訓練模型往特定任務調整 |
| 3 | 分類頭 | 接在 BERT 上的新線性層，輸出各類別的分數 |
| 4 | tokenizer | 把文字轉成模型看得懂的數字，**必須和模型配對** |
| 5 | 動態補齊 | `DataCollatorWithPadding` 只補到 batch 內最長，省運算 |
| 6 | logits | 模型輸出的原始分數，不是機率 |
| 7 | `argmax` | 從分數中選出最大者的索引，當作預測類別 |
| 8 | F1 | precision 與 recall 的調和平均；全猜同一類約 0.667 |

## 四、程式流程速記

```
1. load_dataset → 取 train、test
2. from_pretrained → 載入 BERT + 新分類頭，載入配對的 tokenizer
3. DataCollatorWithPadding + map(tokenizer) → 資料轉成數字，批次時動態補齊
4. compute_metrics → logits 經 argmax 變預測，再算 F1
5. TrainingArguments → 學習率、batch、epoch 等
6. Trainer → 把所有零件組裝起來
7. evaluate（前）→ train → evaluate（後）
```

## 五、最容易搞混的地方

1. **哪一行是微調？**
   沒有單獨一行，是 `from_pretrained` + `num_labels` + `trainer.train()` 的組合；最核心的動作在 `trainer.train()` 內部。

2. **為什麼學習率這麼小（2e-5）？**
   預訓練權重已經很好，只想小幅調整；學習率太大會把學到的知識洗掉。

3. **`evaluate()` 和 `train()` 差在哪？**
   前者只打分數，後者才更新權重。

4. **為什麼分詞不先補齊？**
   留給 `DataCollatorWithPadding` 在組 batch 時才補，只補到該 batch 最長，比固定補到最大長度省。

5. **模型輸出為什麼要 `argmax`？**
   模型輸出的是 2 個分數，F1 要的是類別（0 或 1）。

6. **看到 "newly initialized" 警告要緊張嗎？**
   不用，那是在告訴你分類頭是新的，等訓練去學。

## 六、觀察到的幾個地方（程式碼未修改，僅供參考）

這些不一定是錯誤，是進階時可以思考的點：

1. **用 test 集做評估**
   資料集有 `validation` 集。標準做法是：訓練中的評估用 validation，**test 留到最後只看一次**，避免對 test 集做了太多決策而產生樂觀偏差。練習時直接用 test 沒問題。

2. **訓練中不會評估**
   沒有設定 `eval_strategy`，所以只有訓練前後各評估一次，看不到訓練過程中的分數變化。

3. **沒有儲存最終模型**
   `save_strategy="epoch"` 只存 checkpoint。要保存最終模型，要另外呼叫 `trainer.save_model("某個資料夾")`。

4. **tokenizer 沒有傳進 Trainer**
   存檔時不會連同 tokenizer 一起存，之後載入模型要另外載入 tokenizer。（這個參數在不同版本的 transformers 名稱不同，我沒有確認你的版本。）

5. **`evaluate.load("f1")` 放在函式內**
   每次評估都重新載入一次，移到函式外面只載入一次更乾淨。

6. **沒有固定隨機種子**
   分類頭是隨機初始化，每次跑結果會略有不同。需要可重現時，在 `TrainingArguments` 設定 `seed`。

7. **只訓練 1 個 epoch**
   微調 BERT 常見是 2~4 個 epoch，可以試試看分數是否繼續進步。

8. **沒有開 `fp16`**
   有 GPU 時開 `fp16 = True` 能加速並省記憶體。

## 七、可以自己做的實驗（需要時再自己加，不影響原程式）

```python
# 1. 驗證權重真的變了：在 trainer.train() 前後各取一份比較
import copy
before = copy.deepcopy(model.state_dict())
trainer.train()
after = model.state_dict()
for name in ["classifier.weight", "bert.encoder.layer.0.attention.self.query.weight"]:
    print(name, (before[name] - after[name]).abs().mean().item())

# 2. 比較「全參數微調」與「只訓練分類頭」：凍結 BERT 本體
for p in model.bert.parameters():
    p.requires_grad = False

# 3. 訓練後保存並使用模型
trainer.save_model("my_sentiment_model")
tokenizer.save_pretrained("my_sentiment_model")

from transformers import pipeline
clf = pipeline("text-classification", model="my_sentiment_model")
print(clf("an unforgettable and moving film"))
```

## 八、與 sentence-transformers 練習的對照

| | 這份（Trainer 微調分類器） | 前面的 sentence-transformers 練習 |
|---|---|---|
| 起點 | 預訓練 BERT | 預訓練 BERT |
| 最終產物 | **分類器**（輸出類別） | **句向量模型**（輸出向量） |
| 任務頭 | 新的分類層 | Pooling 層 |
| 損失函數 | 交叉熵（內建） | Softmax、Cosine、MNR、DAE 等，可選 |
| 資料格式 | 文字 + 單一標籤 | 句子對、三元組等 |
| 評估 | F1 | STS-B 相似度相關係數 |

兩者共通的主軸：**從預訓練權重出發，用任務資料與損失函數，呼叫 `train()` 更新權重**。

## 九、一句話記住

> **微調 = 預訓練的起點（`from_pretrained`）＋ 新的任務頭（`num_labels`）＋ 小學習率訓練（`trainer.train()`）。**